In [1]:
# ==========================================================
# NAIVE BAYES FROM SCRATCH EN PYTHON
# ==========================================================
#
# NAIVE BAYES :
# ----------------------------------------------------------
# Algorithme de Machine Learning supervisé
# utilisé pour la classification.
#
# ----------------------------------------------------------
# EXEMPLES D'UTILISATION :
# ----------------------------------------------------------
#
# ✔ Détection spam
# ✔ Analyse sentiments
# ✔ Classification texte
# ✔ Diagnostic médical
#
# ----------------------------------------------------------
# PRINCIPE :
# ----------------------------------------------------------
#
# L'algorithme utilise :
#
# -> le Théorème de Bayes
#
# pour calculer :
#
# P(Classe | Données)
#
# c'est-à-dire :
#
# probabilité qu'un exemple
# appartienne à une classe.
#
# ----------------------------------------------------------
# POURQUOI "NAIVE" ?
# ----------------------------------------------------------
#
# Car l'algorithme suppose que :
#
# toutes les features sont
# indépendantes entre elles.
#
# Exemple :
#
# le mot "Free"
# et
# le mot "Win"
#
# sont considérés indépendants.
#
# Même si dans la réalité
# ce n'est pas totalement vrai.
#
# ----------------------------------------------------------
# FORMULE PRINCIPALE :
# ----------------------------------------------------------
#
# P(A|B) =
#
# P(B|A) * P(A)
# ----------------
#      P(B)
#
# ----------------------------------------------------------
# DANS CE CODE :
# ----------------------------------------------------------
#
# ✔ calcul probabilités classes
# ✔ calcul probabilités features
# ✔ classification emails spam
#
#
# ==========================================================

# ==========================================================
# IMPORTATION DES LIBRARIES
# ==========================================================

# pandas :
# manipulation tableaux/données
import pandas as pd

# numpy :
# calculs mathématiques
import numpy as np

# ==========================================================
# 1. DATASET
# ==========================================================

# ----------------------------------------------------------
# Exemple :
# Détection de spam
# ----------------------------------------------------------
#
# Chaque colonne représente :
#
# 1 = mot présent
# 0 = mot absent
#
# ----------------------------------------------------------
# FEATURES :
# ----------------------------------------------------------
#
# Free
# Win
# Money
# Urgent
#
# ----------------------------------------------------------
# TARGET :
# ----------------------------------------------------------
#
# Spam :
#
# Yes = spam
# No  = non spam
#
# ==========================================================

data = {

    'Free':     [1, 1, 1, 0, 0, 0, 1, 0],

    'Win':      [1, 1, 0, 0, 0, 1, 1, 0],

    'Money':    [1, 0, 1, 0, 0, 1, 1, 0],

    'Urgent':   [1, 0, 1, 0, 0, 0, 1, 0],

    'Spam':     [

        'Yes',

        'Yes',

        'Yes',

        'No',

        'No',

        'No',

        'Yes',

        'No'
    ]
}

# ----------------------------------------------------------
# Transformer dictionnaire en DataFrame
# ----------------------------------------------------------

df = pd.DataFrame(data)

# ----------------------------------------------------------
# Afficher dataset
# ----------------------------------------------------------

print("===== DATASET =====")

print(df)

# ==========================================================
# 2. CLASSE NAIVE BAYES
# ==========================================================

class NaiveBayes:

    """
    ------------------------------------------------------
    Classe principale Naive Bayes.
    ------------------------------------------------------

    Elle contient :

    ✔ entraînement
    ✔ calcul probabilités
    ✔ prédictions
    """

    def __init__(self):

        # --------------------------------------------------
        # Probabilités des classes
        # --------------------------------------------------
        #
        # Exemple :
        #
        # P(Spam=Yes)
        # P(Spam=No)
        #
        # --------------------------------------------------

        self.class_probs = {}

        # --------------------------------------------------
        # Probabilités des features
        # --------------------------------------------------
        #
        # Exemple :
        #
        # P(Free=1 | Spam=Yes)
        #
        # --------------------------------------------------

        self.feature_probs = {}

        # --------------------------------------------------
        # Liste des classes
        # --------------------------------------------------

        self.classes = None

    # ======================================================
    # ENTRAINEMENT
    # ======================================================

    def fit(self, X, y):

        """
        --------------------------------------------------
        Fonction d'entraînement.
        --------------------------------------------------

        Objectif :

        calculer toutes les probabilités
        nécessaires pour Naive Bayes.
        """

        # --------------------------------------------------
        # Classes uniques
        # --------------------------------------------------
        #
        # Exemple :
        #
        # ['Yes', 'No']
        #
        # --------------------------------------------------

        self.classes = np.unique(y)

        # --------------------------------------------------
        # Nombre total exemples
        # --------------------------------------------------

        total_samples = len(y)

        # ==================================================
        # CALCUL P(CLASSE)
        # ==================================================
        #
        # Exemple :
        #
        # P(Spam=Yes)
        #
        # = nombre Yes / total
        #
        # ==================================================

        for c in self.classes:

            # ------------------------------------------------
            # Nombre exemples classe c
            # ------------------------------------------------

            class_count = np.sum(y == c)

            # ------------------------------------------------
            # Calcul probabilité classe
            # ------------------------------------------------

            self.class_probs[c] = (

                class_count / total_samples
            )

        # --------------------------------------------------
        # Affichage probabilités classes
        # --------------------------------------------------

        print("\n===== P(CLASSE) =====")

        for c in self.class_probs:

            print(

                f"{c} : "

                f"{self.class_probs[c]:.4f}"
            )

        # ==================================================
        # CALCUL P(FEATURE | CLASSE)
        # ==================================================
        #
        # Exemple :
        #
        # P(Free=1 | Spam=Yes)
        #
        # ==================================================

        for c in self.classes:

            # ------------------------------------------------
            # Données appartenant à classe c
            # ------------------------------------------------

            X_c = X[y == c]

            # ------------------------------------------------
            # Dictionnaire probabilités
            # ------------------------------------------------

            self.feature_probs[c] = {}

            # ------------------------------------------------
            # Parcourir chaque feature
            # ------------------------------------------------

            for feature in X.columns:

                # =================================================
                # LAPLACE SMOOTHING
                # =================================================
                #
                # Formule :
                #
                # (nombre + 1)
                # ----------------
                # (total + 2)
                #
                # -------------------------------------------------
                # Pourquoi ?
                # -------------------------------------------------
                #
                # éviter probabilité = 0
                #
                # car :
                #
                # multiplication par 0
                # détruit calcul final.
                #
                # =================================================

                prob = (

                    np.sum(X_c[feature]) + 1

                ) / (

                    len(X_c) + 2
                )

                # ------------------------------------------------
                # Sauvegarder probabilité
                # ------------------------------------------------

                self.feature_probs[c][feature] = prob

        # --------------------------------------------------
        # Affichage probabilités features
        # --------------------------------------------------

        print("\n===== P(FEATURE | CLASSE) =====")

        for c in self.feature_probs:

            print(f"\nClasse : {c}")

            for feature in self.feature_probs[c]:

                print(

                    f"{feature} : "

                    f"{self.feature_probs[c][feature]:.4f}"
                )

    # ======================================================
    # PRÉDICTION D'UNE SEULE LIGNE
    # ======================================================

    def predict_one(self, x):

        """
        --------------------------------------------------
        Faire prédiction pour un seul exemple.
        --------------------------------------------------

        On calcule :

        score de chaque classe.

        Puis on choisit :
        la plus grande probabilité.
        """

        # --------------------------------------------------
        # Dictionnaire probabilités finales
        # --------------------------------------------------

        posteriors = {}

        # ==================================================
        # CALCUL POUR CHAQUE CLASSE
        # ==================================================

        for c in self.classes:

            # ------------------------------------------------
            # Commencer avec P(classe)
            # ------------------------------------------------
            #
            # utilisation log :
            #
            # éviter très petits nombres
            #
            # ------------------------------------------------

            posterior = np.log(

                self.class_probs[c]
            )

            # =================================================
            # MULTIPLIER PROBABILITÉS FEATURES
            # =================================================

            for feature in x.index:

                # ------------------------------------------------
                # récupérer probabilité feature
                # ------------------------------------------------

                prob = self.feature_probs[c][feature]

                # =================================================
                # CAS FEATURE = 1
                # =================================================

                if x[feature] == 1:

                    posterior += np.log(prob)

                # =================================================
                # CAS FEATURE = 0
                # =================================================

                else:

                    posterior += np.log(1 - prob)

            # ------------------------------------------------
            # Sauvegarder score final
            # ------------------------------------------------

            posteriors[c] = posterior

        # ==================================================
        # CHOISIR MEILLEURE CLASSE
        # ==================================================
        #
        # classe avec plus grand score
        #
        # ==================================================

        return max(

            posteriors,

            key=posteriors.get
        )

    # ======================================================
    # PRÉDICTION PLUSIEURS EXEMPLES
    # ======================================================

    def predict(self, X):

        """
        --------------------------------------------------
        Faire prédictions pour plusieurs lignes.
        --------------------------------------------------
        """

        predictions = []

        # --------------------------------------------------
        # Parcourir chaque ligne
        # --------------------------------------------------

        for _, row in X.iterrows():

            # ------------------------------------------------
            # prédiction ligne actuelle
            # ------------------------------------------------

            prediction = self.predict_one(row)

            # ------------------------------------------------
            # ajouter résultat
            # ------------------------------------------------

            predictions.append(prediction)

        # --------------------------------------------------
        # retourner toutes prédictions
        # --------------------------------------------------

        return predictions

# ==========================================================
# 3. PRÉPARATION DONNÉES
# ==========================================================

# ----------------------------------------------------------
# X :
# variables explicatives
# ----------------------------------------------------------

X = df.drop(columns=['Spam'])

# ----------------------------------------------------------
# y :
# variable cible
# ----------------------------------------------------------

y = df['Spam']

# ==========================================================
# 4. ENTRAINEMENT
# ==========================================================

# ----------------------------------------------------------
# Création modèle
# ----------------------------------------------------------

model = NaiveBayes()

# ----------------------------------------------------------
# Entraînement modèle
# ----------------------------------------------------------

model.fit(X, y)

# ==========================================================
# 5. TEST
# ==========================================================

# ----------------------------------------------------------
# Nouvel email à classifier
# ----------------------------------------------------------
#
# Free = 1
# Win = 1
# Money = 0
# Urgent = 1
#
# ----------------------------------------------------------

test_email = pd.DataFrame({

    'Free': [1],

    'Win': [1],

    'Money': [0],

    'Urgent': [1]
})

# ----------------------------------------------------------
# Faire prédiction
# ----------------------------------------------------------

prediction = model.predict(test_email)

# ==========================================================
# AFFICHAGE RÉSULTAT
# ==========================================================

print("\n===== TEST =====")

print(test_email)

print("\nClasse prédite :", prediction[0])

# ----------------------------------------------------------
# Exemple résultat :
# ----------------------------------------------------------
#
# Yes -> Spam
# No  -> Non Spam
#
# ----------------------------------------------------------

===== DATASET =====
   Free  Win  Money  Urgent Spam
0     1    1      1       1  Yes
1     1    1      0       0  Yes
2     1    0      1       1  Yes
3     0    0      0       0   No
4     0    0      0       0   No
5     0    1      1       0   No
6     1    1      1       1  Yes
7     0    0      0       0   No

===== P(CLASSE) =====
No : 0.5000
Yes : 0.5000

===== P(FEATURE | CLASSE) =====

Classe : No
Free : 0.1667
Win : 0.3333
Money : 0.3333
Urgent : 0.1667

Classe : Yes
Free : 0.8333
Win : 0.6667
Money : 0.6667
Urgent : 0.6667

===== TEST =====
   Free  Win  Money  Urgent
0     1    1      0       1

Classe prédite : Yes
